In [3]:
import sys
!{sys.executable} -m pip install duckdb

In [4]:
import pandas as pd
import numpy as np
import json
import os
import sys
import time
import datetime
import re
import duckdb

In [5]:
d1= pd.read_json('/Users/abhinav/Desktop/finops_platform/data/customers_raw.json')
d2 = pd.read_csv('/Users/abhinav/Desktop/finops_platform/data/invoices.csv')
d3 = pd.read_csv('/Users/abhinav/Desktop/finops_platform/data/product_usage.csv')
d4 = pd.read_csv('/Users/abhinav/Desktop/finops_platform/data/subscriptions.csv')
d5 = pd.read_csv('/Users/abhinav/Desktop/finops_platform/data/transactions.csv')
d6 = pd.read_csv('/Users/abhinav/Desktop/finops_platform/data/support_tickets.csv')


In [6]:
customers_raw = pd.DataFrame(d1)
invoices = pd.DataFrame(d2)
product_usage = pd.DataFrame(d3)
subscriptions = pd.DataFrame(d4)
transactions = pd.DataFrame(d5)
support_tickets = pd.DataFrame(d6)




In [7]:
duckdb.sql("SELECT * FROM customers_raw LIMIT 10").df()

,customer_id,name,email,phone,address,signup_date,is_active,company,loyalty_tier
0,CUST-0224,"{'first': 'Kenneth', 'last': 'Young'}",kenneth_young795@protonmail.com,+33-556-6877,"{'city': 'Hong Kong', 'country': 'China', 'zip...",30/03/2022,true,CloudNine,Bronze
1,CUST-0449,"{'first': 'Aiko', 'last': 'Garcia'}",aiko_garcia168@company.com,+49-880-2986,"{'city': 'Berlin', 'country': 'Germany', 'zip'...",19/12/2025,yes,AutoDrive,Gold
2,CUST-0112,"{'first': 'Carlos', 'last': 'Campbell'}",carlos.campbell@outlook.com,+81-552-2904,"{'city': 'Dubai', 'country': 'UAE', 'zip': '86...",03/01/2024 00:00,False,NaN,PLATINUM
3,CUST-0130,"{'first': 'Luis', 'last': 'roberts'}",luis_roberts@outlook.com,+49-559-9395,"{'city': 'San Francisco', 'country': 'US', 'zi...",09/01/2023,False,CloudNine,SILVER
4,CUST-0410,"{'first': 'Omar', 'last': 'Ali'}",omar_ali133@outlook.com,+61-221-7694,"{'city': 'Singapore', 'country': 'Singapore', ...",13-Dec-2024,no,GreenEnergy,bronze
5,CUST-0302,"{'first': 'Ahmed', 'last': 'Anderson'}",ahmedanderson230@yahoo.com,+44-433-2162,"{'city': 'Chicago', 'country': 'US', 'zip': '7...",09/08/2023 00:00,1,SecureNet,platinum
6,CUST-0476,"{'first': 'Mark', 'last': 'Zhang'}",markzhang5@hotmail.com,+44-380-8836,"{'city': 'Abu Dhabi', 'country': 'UAE', 'zip':...",07/23/2023 00:00,True,TravelEase,bronze
7,CUST-0478,"{'first': 'Brian', 'last': 'Khan'}",brian.khan976@icloud.com,+61-870-1886,"{'city': 'Austin', 'country': 'US', 'zip': '49...",05/28/2021 00:00,Y,HealthPlus,gold
8,CUST-0223,"{'first': 'Kevin', 'last': 'Rivera'}",kevin.rivera41@protonmail.com,+65-810-3496,"{'city': 'Singapore', 'country': 'Singapore', ...",06-Oct-2024,1,HealthPlus,bronze
9,CUST-0106,"{'first': 'Richard', 'last': 'Kumar'}",richardkumar516@mail.com,+91-874-5706,"{'city': 'San Francisco', 'country': 'US', 'zi...",2025-01-09,1,CloudNine,platinum


In [21]:
duckdb.sql("""
    SELECT 
        customer_id,
        strftime('%Y-%m', 
            COALESCE(
                try_strptime(signup_date, '%d/%m/%Y %H:%M'),
                try_strptime(signup_date, '%d/%m/%Y')
            )
        ) AS cohort_month,
        COUNT(customer_id) OVER (
            PARTITION BY strftime('%Y-%m', 
                COALESCE(
                    try_strptime(signup_date, '%d/%m/%Y %H:%M'),
                    try_strptime(signup_date, '%d/%m/%Y')
                )
            )
        ) AS cohort_size
    FROM customers_raw
 limit 10""").df()

,customer_id,cohort_month,cohort_size
0,CUST-0062,2021-01,1
1,CUST-0300,2021-06,1
2,CUST-0475,2021-07,1
3,CUST-0397,2022-09,1
4,CUST-0291,2022-12,1
5,CUST-0387,2023-05,1
6,CUST-0465,2024-12,4
7,CUST-0164,2024-12,4
8,CUST-0069,2024-12,4
9,CUST-0164,2024-12,4


In [8]:
duckdb.sql("""
    select * from invoices
""").df()

,invoice_id,customer_id,subscription_id,issue_date,due_date,subtotal,tax,total,paid_amount,payment_status,paid_date,payment_method,currency
0,INV-01997,CUST-0495,SUB-00711,10-Apr-2025,2025-05-10T00:00:00,USD 138.12,USD 24.86,USD 162.98,USD 162.98,paid,2025/05/09,Credit Card,AED
1,INV-01033,CUST-0402,SUB-00075,21/06/2022,07-21-2022,USD 311.39,USD 24.91,USD 336.30,USD 336.30,paid,2022/07/19,Credit Card,USD
2,INV-01154,CUST-0114,SUB-00231,04/03/2023 00:00,05-03-2023,$197.82,$0.00,$197.82,0.00,UNPAID,NaN,paypal,GBP
3,INV-02877,CUST-0131,SUB-00277,09/12/2024 00:00,10-12-2024,$ 826.53,$ 41.33,$ 867.86,$ 867.86,paid,01/11/2024,wire,USD
4,INV-02233,CUST-0039,SUB-00245,2023/03/08,03/23/2023 00:00,$49.43,$4.94,$54.37,$22.61,partial,2023-04-12T00:00:00,Credit Card,USD
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3035,INV-01985,CUST-0195,SUB-00316,2024-05-23,22/06/2024,459.60,59.75,519.35,519.35,paid,06-Aug-2024,bank_transfer,USD
3036,INV-00385,CUST-0435,SUB-00522,"Apr 04, 2023",2023-06-03T00:00:00,$752.86,$150.57,$903.43,$903.43,paid,06-18-2023,PAYPAL,USD
3037,INV-00199,CUST-0245,SUB-00742,02-18-2024,2024-03-04T00:00:00,AED 846.23,AED 110.01,AED 956.24,AED 956.24,paid,2024-03-05,credit_card,GBP
3038,INV-02909,CUST-0168,SUB-00375,2022-09-15T00:00:00,15-Oct-2022,542.36,108.47,650.83,650.83,paid,01/13/2023 00:00,PAYPAL,USD


In [10]:
duckdb.sql("""
    select * from product_usage limit 10
""").df()

,usage_id,customer_id,feature_name,session_date,session_duration_seconds,usage_count,device,session_id
0,USG-008226,CUST-0213,sandbox,07-22-2024,5502,127,MOBILE,SES-467409
1,USG-006508,CUST-0472,dashboard,03-Nov-2025,921,45,MOBILE,SES-634317
2,USG-005797,CUST-0317,custom_reports,05/11/2025 00:00,921,27,desktop,SES-784576
3,USG-005873,CUST-0049,analytics,"May 28, 2025",2170,175,api,SES-926991
4,USG-005053,CUST-0406,analytics,2025/07/02,5656,123,web,SES-153593
5,USG-001563,CUST-0299,dashboard,2024-03-25T00:00:00,980,27,API,SES-656106
6,USG-000064,CUST-0329,sandbox,02/28/2025 00:00,4101,40,Desktop,SES-258231
7,USG-003943,CUST-0098,anaytics,2024/05/09,1682,98,Web,SES-953889
8,USG-000542,CUST-0092,bulk_export,2025/07/12,897,126,NaN,SES-328778
9,USG-004197,CUST-0353,api_access,12/17/2025 00:00,3036,94,Web,SES-437089


In [13]:
duckdb.sql("""
WITH cohort_base AS (
    SELECT
        customer_id,
        strftime(
            COALESCE(
                try_strptime(signup_date, '%d/%m/%Y'),
                try_strptime(signup_date, '%Y-%m-%d'),
                try_strptime(signup_date, '%m/%d/%Y'),
                try_strptime(signup_date, '%Y/%m/%d')
            ),
            '%Y-%m'
        ) AS cohort_month,
        COUNT(customer_id) OVER (
            PARTITION BY strftime(
                COALESCE(
                    try_strptime(signup_date, '%d/%m/%Y'),
                    try_strptime(signup_date, '%Y-%m-%d'),
                    try_strptime(signup_date, '%m/%d/%Y'),
                    try_strptime(signup_date, '%Y/%m/%d')
                ),
                '%Y-%m'
            )
        ) AS cohort_size
    FROM customers_raw
    WHERE signup_date IS NOT NULL
),

txn_enriched AS (
    SELECT
        t.customer_id,
        t.amount_clean AS amount,
        cb.cohort_month,
        cb.cohort_size,
        CAST(
            date_diff(
                'month',
                strptime(cb.cohort_month, '%Y-%m'),
                date_trunc(
                    'month',
                    COALESCE(
                        try_strptime(t.transaction_date, '%Y-%m-%d'),
                        try_strptime(t.transaction_date, '%Y/%m/%d'),
                        try_strptime(t.transaction_date, '%d/%m/%Y'),
                        try_strptime(t.transaction_date, '%b %d, %Y'),
                        try_strptime(t.transaction_date, '%B %d, %Y')
                    )
                )
            ) + 1
        AS INTEGER) AS months_since_signup
    FROM (
        -- inline amount cleaning: strip currency symbols
        SELECT
            customer_id,
            transaction_date,
            status,
            TRY_CAST(
                regexp_replace(amount, '[^0-9.]', '', 'g')
            AS DOUBLE) AS amount_clean
        FROM transactions
        WHERE status = 'completed'
          AND transaction_date IS NOT NULL
    ) t
    JOIN cohort_base cb ON t.customer_id = cb.customer_id
    WHERE amount_clean > 0
),

cohort_revenue AS (
    SELECT
        cohort_month,
        MAX(cohort_size)                                         AS cohort_size,

        ROUND(SUM(CASE WHEN months_since_signup = 1  THEN amount ELSE 0 END), 2) AS rev_m1,
        ROUND(SUM(CASE WHEN months_since_signup = 2  THEN amount ELSE 0 END), 2) AS rev_m2,
        ROUND(SUM(CASE WHEN months_since_signup = 3  THEN amount ELSE 0 END), 2) AS rev_m3,
        ROUND(SUM(CASE WHEN months_since_signup = 6  THEN amount ELSE 0 END), 2) AS rev_m6,
        ROUND(SUM(CASE WHEN months_since_signup = 12 THEN amount ELSE 0 END), 2) AS rev_m12,

        COUNT(DISTINCT CASE WHEN months_since_signup = 1  THEN customer_id END) AS retained_m1,
        COUNT(DISTINCT CASE WHEN months_since_signup = 2  THEN customer_id END) AS retained_m2,
        COUNT(DISTINCT CASE WHEN months_since_signup = 3  THEN customer_id END) AS retained_m3,
        COUNT(DISTINCT CASE WHEN months_since_signup = 6  THEN customer_id END) AS retained_m6,
        COUNT(DISTINCT CASE WHEN months_since_signup = 12 THEN customer_id END) AS retained_m12
    FROM txn_enriched
    GROUP BY cohort_month
)

SELECT
    cohort_month,
    cohort_size,
    rev_m1,   rev_m2,   rev_m3,   rev_m6,   rev_m12,
    ROUND(100.0 * retained_m1  / NULLIF(cohort_size, 0), 1) AS retention_pct_m1,
    ROUND(100.0 * retained_m2  / NULLIF(cohort_size, 0), 1) AS retention_pct_m2,
    ROUND(100.0 * retained_m3  / NULLIF(cohort_size, 0), 1) AS retention_pct_m3,
    ROUND(100.0 * retained_m6  / NULLIF(cohort_size, 0), 1) AS retention_pct_m6,
    ROUND(100.0 * retained_m12 / NULLIF(cohort_size, 0), 1) AS retention_pct_m12
FROM cohort_revenue
WHERE cohort_month IS NOT NULL
ORDER BY cohort_month

""").df()

,cohort_month,cohort_size,rev_m1,rev_m2,rev_m3,rev_m6,rev_m12,retention_pct_m1,retention_pct_m2,retention_pct_m3,retention_pct_m6,retention_pct_m12
0,2021-01,2,0.00,0.00,0.00,0.00,0.00,0.0,0.0,0.0,0.0,0.0
1,2021-02,1,0.00,0.00,0.00,0.00,0.00,0.0,0.0,0.0,0.0,0.0
2,2021-03,7,0.00,0.00,0.00,0.00,0.00,0.0,0.0,0.0,0.0,0.0
3,2021-04,2,0.00,0.00,0.00,0.00,0.00,0.0,0.0,0.0,0.0,0.0
4,2021-05,4,0.00,0.00,0.00,0.00,575.80,0.0,0.0,0.0,0.0,25.0
5,2021-06,3,0.00,0.00,0.00,0.00,0.00,0.0,0.0,0.0,0.0,0.0
6,2021-07,3,0.00,0.00,0.00,0.00,0.00,0.0,0.0,0.0,0.0,0.0
7,2021-08,2,0.00,0.00,0.00,0.00,0.00,0.0,0.0,0.0,0.0,0.0
8,2021-09,5,0.00,0.00,0.00,0.00,0.00,0.0,0.0,0.0,0.0,0.0
9,2021-10,6,0.00,0.00,0.00,1101.78,0.00,0.0,0.0,0.0,16.7,0.0


In [14]:
duckdb.sql("""
WITH cleaned_subs AS (
    -- Step 1: clean the subscriptions table
    -- Fix mixed date formats, negative MRR, and normalise status casing
    SELECT
        subscription_id,
        customer_id,
        LOWER(TRIM(status))                                      AS status,
        ABS(TRY_CAST(mrr AS DOUBLE))                            AS mrr,
        COALESCE(
            try_strptime(start_date, '%Y-%m-%dT%H:%M:%S'),
            try_strptime(start_date, '%Y-%m-%d'),
            try_strptime(start_date, '%d/%m/%Y'),
            try_strptime(start_date, '%d-%b-%Y'),
            try_strptime(start_date, '%d-%m-%Y'),
            try_strptime(start_date, '%Y/%m/%d')
        )::DATE                                                  AS start_date,
        COALESCE(
            try_strptime(end_date, '%Y-%m-%dT%H:%M:%S'),
            try_strptime(end_date, '%Y-%m-%d'),
            try_strptime(end_date, '%d/%m/%Y'),
            try_strptime(end_date, '%d-%b-%Y'),
            try_strptime(end_date, '%d-%m-%Y'),
            try_strptime(end_date, '%Y/%m/%d')
        )::DATE                                                  AS end_date
    FROM subscriptions
    WHERE start_date IS NOT NULL
      AND mrr IS NOT NULL
),

month_spine AS (
    -- Step 2: generate one row for every month between earliest and latest date
    -- This is DuckDB's equivalent of joining to date_dim
    -- generate_series produces a sequence of timestamps, we cast to date
    SELECT
        date_trunc('month', gs.month_start)::DATE AS month_start
    FROM (
        SELECT unnest(
            generate_series(
                (SELECT date_trunc('month', MIN(start_date)) FROM cleaned_subs),
                (SELECT date_trunc('month', MAX(COALESCE(end_date, CURRENT_DATE)))) ,
                INTERVAL '1 month'
            )
        ) AS month_start
    ) gs
),

monthly_mrr AS (
    -- Step 3: for each customer, sum MRR for every month their sub was active
    -- Equivalent of the date_dim JOIN with day_of_month = 1 in SQLite
    SELECT
        s.customer_id,
        strftime(m.month_start, '%Y-%m')                         AS month,
        SUM(s.mrr)                                               AS mrr
    FROM cleaned_subs s
    JOIN month_spine m
        ON  m.month_start >= date_trunc('month', s.start_date)
        AND m.month_start <= date_trunc('month', COALESCE(s.end_date, '2099-12-31'::DATE))
    WHERE s.status IN ('active', 'cancelled', 'expired')
    GROUP BY s.customer_id, strftime(m.month_start, '%Y-%m')
),

mrr_with_lag AS (
    -- Step 4: use LAG() to bring in the previous month's MRR per customer
    SELECT
        customer_id,
        month,
        mrr                                                      AS curr_mrr,
        LAG(mrr, 1, 0) OVER (
            PARTITION BY customer_id
            ORDER BY month
        )                                                        AS prev_mrr
    FROM monthly_mrr
),

mrr_classified AS (
    -- Step 5: classify each row as new / expansion / contraction / churned
    SELECT
        month,
        customer_id,
        curr_mrr,
        prev_mrr,
        curr_mrr - prev_mrr                                      AS mrr_delta,
        CASE
            WHEN prev_mrr = 0   AND curr_mrr > 0               THEN 'new'
            WHEN prev_mrr > 0   AND curr_mrr > prev_mrr        THEN 'expansion'
            WHEN prev_mrr > 0   AND curr_mrr < prev_mrr
                                AND curr_mrr > 0               THEN 'contraction'
            WHEN prev_mrr > 0   AND curr_mrr = 0               THEN 'churned'
            ELSE                                                     'unchanged'
        END                                                      AS movement_type
    FROM mrr_with_lag
)

-- Step 6: aggregate by month, pivot movement types into columns
SELECT
    month,
    ROUND(SUM(CASE WHEN movement_type = 'new'
                   THEN curr_mrr  ELSE 0 END), 2)               AS new_mrr,
    ROUND(SUM(CASE WHEN movement_type = 'expansion'
                   THEN mrr_delta ELSE 0 END), 2)               AS expansion_mrr,
    ROUND(SUM(CASE WHEN movement_type = 'contraction'
                   THEN mrr_delta ELSE 0 END), 2)               AS contraction_mrr,
    ROUND(SUM(CASE WHEN movement_type = 'churned'
                   THEN -prev_mrr ELSE 0 END), 2)               AS churned_mrr,
    ROUND(SUM(mrr_delta), 2)                                     AS net_new_mrr,
    -- running total of MRR across all months
    ROUND(SUM(SUM(curr_mrr)) OVER (ORDER BY month), 2)          AS cumulative_mrr
FROM mrr_classified
WHERE movement_type != 'unchanged'
GROUP BY month
ORDER BY month

""").df()

BinderException: Binder Error: Referenced column "end_date" was not found because the FROM clause is missing